In [1]:
import pandas as pd

# 讀取 csv 檔案
df = pd.read_csv('Sample-Superstore.csv', encoding='latin-1')
print(df.columns)
customers = df[['Customer ID', 'Customer Name', 'Segment']]
customers

Index(['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode',
       'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State',
       'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category',
       'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit'],
      dtype='str')


,Customer ID,Customer Name,Segment
0,CG-12520,Claire Gute,Consumer
1,CG-12520,Claire Gute,Consumer
2,DV-13045,Darrin Van Huff,Corporate
3,SO-20335,Sean O'Donnell,Consumer
4,SO-20335,Sean O'Donnell,Consumer
...,...,...,...
9989,TB-21400,Tom Boeckenhauer,Consumer
9990,DB-13060,Dave Brooks,Consumer
9991,DB-13060,Dave Brooks,Consumer
9992,DB-13060,Dave Brooks,Consumer


In [2]:
import pymysql
from configparser import ConfigParser
from create_table import create_superstore_tables


# 讀取 .env 檔案取得資料庫連線資訊
config = ConfigParser()
config.read('../Chapter1/config.ini')

# 建立資料庫連線
connection = pymysql.connect(
    host=config.get('DB', 'host'),
    user=config.get('DB', 'user'),
    password=config.get('DB', 'password'),
    port=config.getint('DB', 'port'),
    cursorclass=pymysql.cursors.DictCursor,
)

with connection.cursor() as cursor:
    # 建立資料庫
    cursor.execute(f"CREATE DATABASE IF NOT EXISTS superstore")

    cursor.execute(f"SHOW DATABASES")
    dbs = cursor.fetchall()
    print(dbs)

    # 建立資料表
    create_superstore_tables("superstore")
    

[{'Database': 'chapter2'}, {'Database': 'chapter3'}, {'Database': 'classicmodels'}, {'Database': 'information_schema'}, {'Database': 'my_database'}, {'Database': 'my_titanic'}, {'Database': 'my_train_titanic'}, {'Database': 'mysql'}, {'Database': 'performance_schema'}, {'Database': 'sakila'}, {'Database': 'social_media_app'}, {'Database': 'superstore'}, {'Database': 'sys'}, {'Database': 'testdb'}, {'Database': 'transaction_test'}, {'Database': 'world'}]
{'Tables_in_superstore': 'customers'}
{'Tables_in_superstore': 'orderdetails'}
{'Tables_in_superstore': 'orders'}
{'Tables_in_superstore': 'products'}


In [3]:
# 建立資料庫連線
connection = pymysql.connect(
    host=config.get('DB', 'host'),
    user=config.get('DB', 'user'),
    password=config.get('DB', 'password'),
    port=config.getint('DB', 'port'),
    cursorclass=pymysql.cursors.DictCursor,
    database='superstore'
)

In [4]:
import pandas as pd

# 讀取 CSV 檔案
df = pd.read_csv('Sample-Superstore.csv', encoding='latin-1')

# 提取客戶欄位的資料
print(df.columns)

# 去除重複的客戶資料
customers = df[['Customer ID', 'Customer Name', 'Segment']]
customers = customers.drop_duplicates()
customers


Index(['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode',
       'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State',
       'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category',
       'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit'],
      dtype='str')


,Customer ID,Customer Name,Segment
0,CG-12520,Claire Gute,Consumer
2,DV-13045,Darrin Van Huff,Corporate
3,SO-20335,Sean O'Donnell,Consumer
5,BH-11710,Brosina Hoffman,Consumer
12,AA-10480,Andrew Allen,Consumer
...,...,...,...
8666,CJ-11875,Carl Jackson,Corporate
9209,RS-19870,Roy Skaria,Home Office
9399,SC-20845,Sung Chung,Consumer
9441,RE-19405,Ricardo Emerson,Consumer


Customers

In [ ]:
# 取得 df 中的客戶資料
customers

# 轉成 list 以便後續寫入
customers_list = customers.values.tolist()
print(customers_list)


# 寫入 Customers 資料表
with connection.cursor() as cursor:
    #檢查 customers table 參數
    cursor.execute("DESC customers;")
    result = cursor.fetchall()
    print(result)

    sql = """
        INSERT INTO customers(customer_id, customer_name, segment)
        VALUES(%s, %s, %s);
    """
    cursor.executemany(sql, customers_list)
    #檢查寫入的數量
    print(cursor.rowcount)



In [13]:
connection.commit() # 確認無誤後 執行 commit

In [5]:
#檢查table的資料是否有被寫入
from chart_data import sql_query
r = sql_query("SELECT * FROM customers limit 5")
print(r)

[{'customer_id': 'AA-10315', 'customer_name': 'Alex Avila', 'segment': 'Consumer'}, {'customer_id': 'AA-10375', 'customer_name': 'Allen Armold', 'segment': 'Consumer'}, {'customer_id': 'AA-10480', 'customer_name': 'Andrew Allen', 'segment': 'Consumer'}, {'customer_id': 'AA-10645', 'customer_name': 'Anna Andreadi', 'segment': 'Consumer'}, {'customer_id': 'AB-10015', 'customer_name': 'Aaron Bergman', 'segment': 'Consumer'}]


Orders 

In [6]:
from datetime import datetime

# 取得 df 中的欄位名稱:
print(df.columns)

#檢查 orders table 參數
with connection.cursor() as cursor:
    cursor.execute("DESC orders;")
    result = cursor.fetchall()
    for row in result:
        print(row['Field'])

# 取得 df 中的訂單資料
orders = df[['Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID']]
orders = orders.drop_duplicates()
print(len(orders))
print(orders.head()) # 檢查


# 將日期格式轉換為 datetime.date
#orders['Order Date'] = pd.to_datetime
orders['Order Date'] = pd.to_datetime(orders['Order Date'], format='%m/%d/%Y').dt.date
orders['Ship Date'] = pd.to_datetime(orders['Ship Date'], format='%m/%d/%Y').dt.date
print(orders.head())

# 轉成 list 以便後續寫入
orders = orders.values.tolist()
print(orders[0])


# 寫入 Orders 資料表
with connection.cursor() as cursor:
    sql = """
        INSERT INTO orders(order_id, order_date, ship_date, ship_mode, customer_id)
        VALUES(%s, %s, %s, %s, %s);
    """
    cursor.executemany(sql, orders)
    #檢查寫入的數量
    print(cursor.rowcount)



Index(['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode',
       'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State',
       'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category',
       'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit'],
      dtype='str')
order_id
order_date
ship_date
ship_mode
customer_id
5009
          Order ID  Order Date   Ship Date       Ship Mode Customer ID
0   CA-2016-152156   11/8/2016  11/11/2016    Second Class    CG-12520
2   CA-2016-138688   6/12/2016   6/16/2016    Second Class    DV-13045
3   US-2015-108966  10/11/2015  10/18/2015  Standard Class    SO-20335
5   CA-2014-115812    6/9/2014   6/14/2014  Standard Class    BH-11710
12  CA-2017-114412   4/15/2017   4/20/2017  Standard Class    AA-10480
          Order ID  Order Date   Ship Date       Ship Mode Customer ID
0   CA-2016-152156  2016-11-08  2016-11-11    Second Class    CG-12520
2   CA-2016-138688  2016-06-12  2016-06-16    Second Class    DV-130

In [7]:
# 檢查無誤 commit
connection.commit()

In [8]:
#檢查table的資料是否有被寫入
from chart_data import sql_query
r = sql_query("SELECT * FROM orders limit 5")
print(r)

[{'order_id': 'CA-2014-100006', 'order_date': datetime.date(2014, 9, 7), 'ship_date': datetime.date(2014, 9, 13), 'ship_mode': 'Standard Class', 'customer_id': 'DK-13375'}, {'order_id': 'CA-2014-100090', 'order_date': datetime.date(2014, 7, 8), 'ship_date': datetime.date(2014, 7, 12), 'ship_mode': 'Standard Class', 'customer_id': 'EB-13705'}, {'order_id': 'CA-2014-100293', 'order_date': datetime.date(2014, 3, 14), 'ship_date': datetime.date(2014, 3, 18), 'ship_mode': 'Standard Class', 'customer_id': 'NF-18475'}, {'order_id': 'CA-2014-100328', 'order_date': datetime.date(2014, 1, 28), 'ship_date': datetime.date(2014, 2, 3), 'ship_mode': 'Standard Class', 'customer_id': 'JC-15340'}, {'order_id': 'CA-2014-100363', 'order_date': datetime.date(2014, 4, 8), 'ship_date': datetime.date(2014, 4, 15), 'ship_mode': 'Standard Class', 'customer_id': 'JM-15655'}]


Products

In [16]:
# 取得 df 中的欄位名稱:
print(df.columns)

#檢查 products table 參數
with connection.cursor() as cursor:
    cursor.execute("DESC products;")
    result = cursor.fetchall()
    for row in result:
        print(row['Field'])

# 取得 df 中的產品資料
products = df[['Product ID', 'Category', 'Sub-Category', 'Product Name']]
products = products.drop_duplicates()
print(len(products))
print(products.head()) # 檢查

# 轉成 list 以便後續寫入
products = products.values.tolist()
print(products[0])


# 寫入 Products 資料表
with connection.cursor() as cursor:
    sql = """
        INSERT INTO products(product_id, category, sub_category, product_name)
        VALUES(%s, %s, %s, %s);
    """
    cursor.executemany(sql, products)
    #檢查寫入的數量
    print(cursor.rowcount)



Index(['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode',
       'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State',
       'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category',
       'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit'],
      dtype='str')
product_id
category
sub_category
product_name
1894
        Product ID         Category Sub-Category  \
0  FUR-BO-10001798        Furniture    Bookcases   
1  FUR-CH-10000454        Furniture       Chairs   
2  OFF-LA-10000240  Office Supplies       Labels   
3  FUR-TA-10000577        Furniture       Tables   
4  OFF-ST-10000760  Office Supplies      Storage   

                                        Product Name  
0                  Bush Somerset Collection Bookcase  
1  Hon Deluxe Fabric Upholstered Stacking Chairs,...  
2  Self-Adhesive Address Labels for Typewriters b...  
3      Bretford CR4500 Series Slim Rectangular Table  
4                     Eldon Fold 'N Roll Cart System  

In [ ]:
# 確認無誤 commit
connection.commit()

In [17]:
#檢查table的資料是否有被寫入
from chart_data import sql_query
r = sql_query("SELECT * FROM orders limit 5")
print(r)

[{'order_id': 'CA-2014-100006', 'order_date': datetime.date(2014, 9, 7), 'ship_date': datetime.date(2014, 9, 13), 'ship_mode': 'Standard Class', 'customer_id': 'DK-13375'}, {'order_id': 'CA-2014-100090', 'order_date': datetime.date(2014, 7, 8), 'ship_date': datetime.date(2014, 7, 12), 'ship_mode': 'Standard Class', 'customer_id': 'EB-13705'}, {'order_id': 'CA-2014-100293', 'order_date': datetime.date(2014, 3, 14), 'ship_date': datetime.date(2014, 3, 18), 'ship_mode': 'Standard Class', 'customer_id': 'NF-18475'}, {'order_id': 'CA-2014-100328', 'order_date': datetime.date(2014, 1, 28), 'ship_date': datetime.date(2014, 2, 3), 'ship_mode': 'Standard Class', 'customer_id': 'JC-15340'}, {'order_id': 'CA-2014-100363', 'order_date': datetime.date(2014, 4, 8), 'ship_date': datetime.date(2014, 4, 15), 'ship_mode': 'Standard Class', 'customer_id': 'JM-15655'}]


OrderDetails

In [23]:
# 取得 df 中的欄位名稱:
print(df.columns)

#檢查 orderdetails table 參數
with connection.cursor() as cursor:
    cursor.execute("DESC orderdetails;")
    result = cursor.fetchall()
    for row in result:
        print(row['Field'])

# 取得 df 中的訂單明細資料
order_details = df[['Row ID', 'Order ID', 'Product ID', 'Sales', 'Quantity', 'Discount', 'Profit']]
order_details = order_details.drop_duplicates()
print(len(order_details))
print(order_details.head()) # 檢查

# 轉成 list 以便後續寫入
order_details = order_details.values.tolist()
print(order_details[0])


# 寫入 OrderDetails 資料表
with connection.cursor() as cursor:
    sql = """
        INSERT INTO orderdetails(row_id, order_id, product_id, sales, quantity, discount, profit)
        VALUES(%s, %s, %s, %s, %s, %s, %s);
    """
    cursor.executemany(sql, order_details)
    #檢查寫入的數量
    print(cursor.rowcount)




Index(['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode',
       'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State',
       'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category',
       'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit'],
      dtype='str')
row_id
order_id
product_id
sales
quantity
discount
profit
9994
   Row ID        Order ID       Product ID     Sales  Quantity  Discount  \
0       1  CA-2016-152156  FUR-BO-10001798  261.9600         2      0.00   
1       2  CA-2016-152156  FUR-CH-10000454  731.9400         3      0.00   
2       3  CA-2016-138688  OFF-LA-10000240   14.6200         2      0.00   
3       4  US-2015-108966  FUR-TA-10000577  957.5775         5      0.45   
4       5  US-2015-108966  OFF-ST-10000760   22.3680         2      0.20   

     Profit  
0   41.9136  
1  219.5820  
2    6.8714  
3 -383.0310  
4    2.5164  
[1, 'CA-2016-152156', 'FUR-BO-10001798', 261.96, 2, 0.0, 41.9136]
9994


In [24]:
# 確認無誤 commit
connection.commit()

In [33]:
#檢查table的資料是否有被寫入
from chart_data import sql_query
r = sql_query("SELECT * FROM orderdetails limit 5")
print(r)

[{'row_id': 1, 'order_id': 'CA-2016-152156', 'product_id': 'FUR-BO-10001798', 'sales': Decimal('261.96'), 'quantity': 2, 'discount': Decimal('0.00'), 'profit': Decimal('41.91')}, {'row_id': 2, 'order_id': 'CA-2016-152156', 'product_id': 'FUR-CH-10000454', 'sales': Decimal('731.94'), 'quantity': 3, 'discount': Decimal('0.00'), 'profit': Decimal('219.58')}, {'row_id': 3, 'order_id': 'CA-2016-138688', 'product_id': 'OFF-LA-10000240', 'sales': Decimal('14.62'), 'quantity': 2, 'discount': Decimal('0.00'), 'profit': Decimal('6.87')}, {'row_id': 4, 'order_id': 'US-2015-108966', 'product_id': 'FUR-TA-10000577', 'sales': Decimal('957.58'), 'quantity': 5, 'discount': Decimal('0.45'), 'profit': Decimal('-383.03')}, {'row_id': 5, 'order_id': 'US-2015-108966', 'product_id': 'OFF-ST-10000760', 'sales': Decimal('22.37'), 'quantity': 2, 'discount': Decimal('0.20'), 'profit': Decimal('2.52')}]


In [37]:
sub_category_query = """
    SELECT sub_category AS label, COUNT(OrderDetails.order_id) AS value
    FROM OrderDetails
    JOIN Products
    ON OrderDetails.product_id = Products.product_id
    WHERE Products.category = %s
    GROUP BY Products.sub_category;
"""
result = sql_query(sub_category_query, 'Furniture')
print(result)

[{'label': 'Bookcases', 'value': 228}, {'label': 'Chairs', 'value': 617}, {'label': 'Furnishings', 'value': 957}, {'label': 'Tables', 'value': 319}]


In [36]:
from pprint import pprint
products_order_details_query = """
    SELECT category, sub_category, product_name, sales, profit
    FROM OrderDetails
    JOIN Products
    ON OrderDetails.product_id = Products.product_id;
"""
result = sql_query(products_order_details_query)
pprint(result)

[{'category': 'Furniture',
  'product_name': 'Bush Birmingham Collection Bookcase, Dark Cherry',
  'profit': Decimal('-117.88'),
  'sales': Decimal('825.17'),
  'sub_category': 'Bookcases'},
 {'category': 'Furniture',
  'product_name': 'Sauder Camden County Barrister Bookcase, Planked Cherry '
                  'Finish',
  'profit': Decimal('-4.84'),
  'sales': Decimal('411.33'),
  'sub_category': 'Bookcases'},
 {'category': 'Furniture',
  'product_name': 'Sauder Camden County Barrister Bookcase, Planked Cherry '
                  'Finish',
  'profit': Decimal('-4.84'),
  'sales': Decimal('411.33'),
  'sub_category': 'Bookcases'},
 {'category': 'Furniture',
  'product_name': 'Sauder Camden County Barrister Bookcase, Planked Cherry '
                  'Finish',
  'profit': Decimal('33.87'),
  'sales': Decimal('241.96'),
  'sub_category': 'Bookcases'},
 {'category': 'Furniture',
  'product_name': 'Sauder Inglewood Library Bookcases',
  'profit': Decimal('-35.91'),
  'sales': Decimal('359